<div dir="rtl" align="right">

# بوستینگ: AdaBoost و گرادیان بوستینگ

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## ایده در یک نگاه
بوستینگ مدل‌های **ضعیف** (کمی بهتر از شانس، مثلاً درخت یک‌برشی یا «کنده») را **پشت سر هم** می‌سازد؛ هر مدل روی اشتباه‌های قبلی تمرکز می‌کند. جمع وزن‌دار آن‌ها یک مدل **قوی** می‌شود.

## AdaBoost گام‌به‌گام
- همه نمونه‌ها وزن برابر $w_i=1/n$ دارند.
- برای $m=1..M$:
- یک کنده با وزن‌ها آموزش بده؛ خطای وزن‌دار $\varepsilon_m$ را حساب کن.
- وزن مدل: $\alpha_m=\frac12\ln\frac{1-\varepsilon_m}{\varepsilon_m}$ (خطای کمتر = رأی بیشتر).
- وزن نمونه‌های **غلط** را زیاد و درست‌ها را کم کن: $w_i\leftarrow w_i e^{-\alpha_m y_ih_m(x_i)}$، سپس نرمال کن.
- پیش‌بینی نهایی: $\text{sign}\big(\sum_m\alpha_mh_m(x)\big)$.

## مثال عددی یک دور
۱۰ نمونه، کنده اول ۳ تا را غلط می‌گوید: $\varepsilon=0.3$ و $\alpha=\frac12\ln\frac{0.7}{0.3}=0.42$.

| نمونه | وزن قبل | ضریب | وزن بعد (نرمال‌شده) |
|---|---|---|---|
| ۷ نمونه درست | 0.1 | $e^{-0.42}=0.65$ | 0.071 |
| ۳ نمونه غلط | 0.1 | $e^{0.42}=1.53$ | 0.167 |

حالا ۳ نمونه غلط در مجموع **نیمی** از وزن را دارند؛ کنده بعدی مجبور است آن‌ها را درست کند.



</div>

In [ ]:
import numpy as np

# One AdaBoost round by hand: 10 samples, 3 mistakes
w = np.full(10, 0.1)
wrong = np.array([1, 1, 1, 0, 0, 0, 0, 0, 0, 0], bool)
eps = w[wrong].sum()
alpha = 0.5 * np.log((1 - eps) / eps)
w = w * np.exp(np.where(wrong, alpha, -alpha))
w /= w.sum()
print("error =", eps, " alpha =", round(alpha, 3))
print("new weights:", w.round(3), " weight of wrong samples:", w[wrong].sum().round(3))

<div dir="rtl" align="right">



> [AdaBoost با ۱، ۵ و ۱۰۰ کنده روی دایره‌ها. مرزهای ساده موازی محور، کنار هم یک دایره می‌سازند.](https://raw.githubusercontent.com/behrooztaheri/learn-python-ai/main/assets/img/lessons/boost_cls.png)

*AdaBoost با ۱، ۵ و ۱۰۰ کنده روی دایره‌ها. مرزهای ساده موازی محور، کنار هم یک دایره می‌سازند.*

[[lab:cls_ada]]

## گرادیان بوستینگ برای طبقه‌بندی
مثل درس رگرسیون، ولی درخت‌ها **گرادیان لاگ‌لاس** را یاد می‌گیرند ($y-p$) و خروجی با سیگموید به احتمال تبدیل می‌شود. `HistGradientBoostingClassifier` سریع‌ترین گزینه sklearn است و XGBoost، LightGBM و CatBoost نسخه‌های مشهور آن‌اند.



</div>

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

X, y = load_breast_cancer(return_X_y=True)
for name, m in [("AdaBoost 200", AdaBoostClassifier(n_estimators=200, random_state=0)),
                ("GradientBoosting", GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=2, random_state=0)),
                ("HistGradientBoosting", HistGradientBoostingClassifier(random_state=0))]:
    print(f"{name:22s} accuracy = {cross_val_score(m, X, y, cv=5).mean():.3f}")

<div dir="rtl" align="right">



| | AdaBoost | گرادیان بوستینگ |
|---|---|---|
| روش تمرکز | وزن دادن به نمونه‌های غلط | آموزش روی گرادیان خطا |
| تابع خطا | نمایی | هر تابع مشتق‌پذیر |
| حساسیت به نویز و داده پرت | زیاد | کمتر |
| کاربرد امروز | آموزشی، داده کوچک | استاندارد داده جدولی |

</div>

<div dir="rtl" align="right">

### تمرین

با staged_score دقت آزمون AdaBoost را بعد از هر کنده روی داده سرطان سینه رسم کن.

</div>

In [ ]:
# Your answer here